# DSC 106 Project 3 — Data Processing


In [9]:
# Step 0: Install dependencies 

import intake
import xarray as xr
import numpy as np
import pandas as pd
import gcsfs

In [10]:
# ============================================================
# STEP 1: Open the Pangeo CMIP6 catalog
# ============================================================
print("Opening Pangeo CMIP6 catalog...")
col = intake.open_esm_datastore(
    "https://storage.googleapis.com/cmip6/pangeo-cmip6.json"
)
print(f"Total entries: {len(col.df)}")

Opening Pangeo CMIP6 catalog...
Total entries: 514818


In [11]:
# ============================================================
# STEP 2: Find models that have tas for ALL 4 experiments
# We need: historical + ssp126 + ssp245 + ssp585
# ============================================================
TARGET_EXPERIMENTS = ['historical', 'ssp126', 'ssp245', 'ssp585']

# Filter to monthly tas, r1i1p1f1 member (most common), gn grid
query = col.search(
    variable_id='tas',
    table_id='Amon',           # monthly atmospheric
    member_id='r1i1p1f1',     # first ensemble member
    grid_label='gn',           # native grid
    experiment_id=TARGET_EXPERIMENTS
)

print(f"\nEntries matching query: {len(query.df)}")
print(query.df[['source_id', 'experiment_id', 'zstore']].head(20))


Entries matching query: 98
        source_id experiment_id  \
0     GISS-E2-1-G    historical   
1     BCC-CSM2-MR    historical   
2          MIROC6    historical   
3        BCC-ESM1    historical   
4      MRI-ESM2-0        ssp245   
5      MRI-ESM2-0    historical   
6     CESM2-WACCM    historical   
7           CESM2    historical   
8     BCC-CSM2-MR        ssp126   
9     BCC-CSM2-MR        ssp245   
10    BCC-CSM2-MR        ssp585   
11    SAM0-UNICON    historical   
12    GISS-E2-1-H    historical   
13        CanESM5        ssp585   
14        CanESM5    historical   
15        CanESM5        ssp245   
16        CanESM5        ssp126   
17  AWI-CM-1-1-MR        ssp126   
18  AWI-CM-1-1-MR        ssp245   
19  AWI-CM-1-1-MR        ssp585   

                                               zstore  
0   gs://cmip6/CMIP6/CMIP/NASA-GISS/GISS-E2-1-G/hi...  
1   gs://cmip6/CMIP6/CMIP/BCC/BCC-CSM2-MR/historic...  
2   gs://cmip6/CMIP6/CMIP/MIROC/MIROC6/historical/...  
3   gs://cmi

In [12]:
# ============================================================
# STEP 3: Find models that have ALL 4 experiments
# ============================================================
model_exp_counts = (
    query.df.groupby('source_id')['experiment_id']
    .nunique()
)
complete_models = model_exp_counts[model_exp_counts == 4].index.tolist()
print(f"\nModels with all 4 experiments: {len(complete_models)}")
print(complete_models)

# Pick up to 8 models for a manageable ensemble
MODELS = complete_models[:3] 
print(f"\nUsing models: {MODELS}")


Models with all 4 experiments: 20
['ACCESS-CM2', 'AWI-CM-1-1-MR', 'BCC-CSM2-MR', 'CAMS-CSM1-0', 'CAS-ESM2-0', 'CESM2-WACCM', 'CMCC-CM2-SR5', 'CMCC-ESM2', 'CanESM5', 'FGOALS-g3', 'FIO-ESM-2-0', 'IITM-ESM', 'MIROC6', 'MPI-ESM1-2-HR', 'MPI-ESM1-2-LR', 'MRI-ESM2-0', 'NESM3', 'NorESM2-LM', 'NorESM2-MM', 'TaiESM1']

Using models: ['ACCESS-CM2', 'AWI-CM-1-1-MR', 'BCC-CSM2-MR']


In [13]:
# ============================================================
# STEP 4: Process each model — compute global annual mean anomaly
# ============================================================
# Pre-industrial baseline: 1850-1900
BASELINE_START = 1850
BASELINE_END   = 1900

# Output years
HIST_START = 1880
PROJ_END   = 2100

def global_mean(ds):
    # Coarsen grid first to reduce data size before computing
    weights = np.cos(np.deg2rad(ds.lat))
    # Only load tas, drop everything else
    ta = ds['tas']
    # Coarsen spatially to ~5 degree resolution before loading
    ta_coarse = ta.coarsen(lat=4, lon=4, boundary='trim').mean()
    weights_coarse = np.cos(np.deg2rad(ta_coarse.lat))
    return ta_coarse.weighted(weights_coarse).mean(dim=['lat', 'lon'])

def to_annual(da):
    """Resample monthly data to annual mean."""
    return da.resample(time='YE').mean()

def get_zarr(zstore):
    fs = gcsfs.GCSFileSystem(token='anon')
    store = gcsfs.GCSMap(zstore, gcs=fs)
    return xr.open_zarr(store, consolidated=True)

results = {}  # {model: {experiment: annual_series}}

for model in MODELS:
    print(f"\nProcessing {model}...")
    results[model] = {}

    for exp in TARGET_EXPERIMENTS:
        row = query.df[
            (query.df['source_id'] == model) &
            (query.df['experiment_id'] == exp)
        ]
        if len(row) == 0:
            print(f"  {exp}: not found, skipping")
            continue

        zstore = row.iloc[0]['zstore']
        try:
            ds = get_zarr(zstore)
            gm = global_mean(ds)           # global mean, monthly
            ann = to_annual(gm)            # annual mean
            ann = ann - 273.15             # K → °C
            results[model][exp] = ann
            print(f"  {exp}: OK ({int(ann.time.dt.year.min())}–{int(ann.time.dt.year.max())})")
        except Exception as e:
            print(f"  {exp}: ERROR — {e}")


Processing ACCESS-CM2...
  historical: OK (1850–2014)


/var/folders/5y/1wpgx5fx0_53thqfv1v5x8tm0000gn/T/ipykernel_99603/4088997075.py:29: SerializationWarning: Unable to decode time axis into full numpy.datetime64[ns] objects, continuing using cftime.datetime objects instead, reason: dates out of range. To silence this warning use a coarser resolution 'time_unit' or specify 'use_cftime=True'.
  return xr.open_zarr(store, consolidated=True)
/var/folders/5y/1wpgx5fx0_53thqfv1v5x8tm0000gn/T/ipykernel_99603/4088997075.py:29: SerializationWarning: Unable to decode time axis into full numpy.datetime64[ns] objects, continuing using cftime.datetime objects instead, reason: dates out of range. To silence this warning use a coarser resolution 'time_unit' or specify 'use_cftime=True'.
  return xr.open_zarr(store, consolidated=True)
/var/folders/5y/1wpgx5fx0_53thqfv1v5x8tm0000gn/T/ipykernel_99603/4088997075.py:29: SerializationWarning: Unable to decode time axis into full numpy.datetime64[ns] objects, continuing using cftime.datetime objects instead, 

  ssp126: OK (2015–2300)
  ssp245: OK (2015–2100)


/var/folders/5y/1wpgx5fx0_53thqfv1v5x8tm0000gn/T/ipykernel_99603/4088997075.py:29: SerializationWarning: Unable to decode time axis into full numpy.datetime64[ns] objects, continuing using cftime.datetime objects instead, reason: dates out of range. To silence this warning use a coarser resolution 'time_unit' or specify 'use_cftime=True'.
  return xr.open_zarr(store, consolidated=True)
/var/folders/5y/1wpgx5fx0_53thqfv1v5x8tm0000gn/T/ipykernel_99603/4088997075.py:29: SerializationWarning: Unable to decode time axis into full numpy.datetime64[ns] objects, continuing using cftime.datetime objects instead, reason: dates out of range. To silence this warning use a coarser resolution 'time_unit' or specify 'use_cftime=True'.
  return xr.open_zarr(store, consolidated=True)
/var/folders/5y/1wpgx5fx0_53thqfv1v5x8tm0000gn/T/ipykernel_99603/4088997075.py:29: SerializationWarning: Unable to decode time axis into full numpy.datetime64[ns] objects, continuing using cftime.datetime objects instead, 

  ssp585: OK (2015–2300)

Processing AWI-CM-1-1-MR...
  historical: OK (1851–2014)
  ssp126: ERROR — Index must be monotonic for resampling
  ssp245: ERROR — Index must be monotonic for resampling
  ssp585: OK (2015–2100)

Processing BCC-CSM2-MR...
  historical: OK (1850–2014)
  ssp126: OK (2015–2100)
  ssp245: OK (2015–2100)
  ssp585: OK (2015–2100)


In [14]:
# ============================================================
# STEP 5: Compute baseline offset per model (1850-1900 mean)
# and compute anomaly
# ============================================================
anomalies = {}  # {model: {experiment: anomaly_series}}

for model, exps in results.items():
    if 'historical' not in exps:
        continue
    hist = exps['historical']

    # Baseline: 1850–1900 annual mean
    baseline_mask = (hist.time.dt.year >= BASELINE_START) & \
                    (hist.time.dt.year <= BASELINE_END)
    baseline_val = float(hist.sel(time=baseline_mask).mean())

    anomalies[model] = {}
    for exp, series in exps.items():
        anom = series - baseline_val
        anomalies[model][exp] = anom
    print(f"{model} baseline ({BASELINE_START}–{BASELINE_END}): {baseline_val:.2f}°C")

ACCESS-CM2 baseline (1850–1900): 13.73°C
AWI-CM-1-1-MR baseline (1850–1900): 13.69°C
BCC-CSM2-MR baseline (1850–1900): 14.72°C


In [15]:
# ============================================================
# STEP 6: Build ensemble DataFrame
# Concatenate all models, align years
# ============================================================
rows = []

for model, exps in anomalies.items():
    # Historical: 1880-2014 (typical CMIP6 historical end)
    if 'historical' in exps:
        hist = exps['historical']
        for t in hist.time:
            yr = int(t.dt.year)
            if HIST_START <= yr <= 2014:
                rows.append({
                    'model': model,
                    'experiment': 'historical',
                    'year': yr,
                    'anomaly': float(hist.sel(time=t))
                })

    # Projections: 2015-2100
    for exp in ['ssp126', 'ssp245', 'ssp585']:
        if exp not in exps:
            continue
        proj = exps[exp]
        for t in proj.time:
            yr = int(t.dt.year)
            if 2015 <= yr <= PROJ_END:
                rows.append({
                    'model': model,
                    'experiment': exp,
                    'year': yr,
                    'anomaly': float(proj.sel(time=t))
                })

df = pd.DataFrame(rows)
print(f"\nRaw rows: {len(df)}")
print(df.head())

KeyboardInterrupt: 

In [ ]:
# ============================================================
# STEP 7: Compute ensemble statistics
# median, 17th pct (likely low), 83rd pct (likely high)
# matching IPCC AR6 likely range definition
# ============================================================

# --- historical.csv ---
hist_df = df[df['experiment'] == 'historical']
hist_out = (
    hist_df.groupby('year')['anomaly']
    .agg(anomaly='median')
    .reset_index()
    .sort_values('year')
)
hist_out['anomaly'] = hist_out['anomaly'].round(3)
hist_out.to_csv('historical.csv', index=False)
print(f"\nhistorical.csv: {len(hist_out)} rows ({hist_out['year'].min()}–{hist_out['year'].max()})")

# --- projections.csv ---
proj_rows = []
for scenario in ['ssp126', 'ssp245', 'ssp585']:
    scen_df = df[df['experiment'] == scenario]
    grp = scen_df.groupby('year')['anomaly']
    for year, group in grp:
        vals = group.values
        proj_rows.append({
            'year': year,
            'scenario': scenario,
            'median':      round(float(np.median(vals)), 3),
            'likely_low':  round(float(np.percentile(vals, 17)), 3),
            'likely_high': round(float(np.percentile(vals, 83)), 3),
        })

proj_out = pd.DataFrame(proj_rows).sort_values(['scenario', 'year'])
proj_out.to_csv('projections.csv', index=False)
print(f"projections.csv: {len(proj_out)} rows")
print(proj_out[proj_out['year'].isin([2030, 2050, 2090])].to_string())

print("\nDone. Copy historical.csv and projections.csv to your project3/data/ folder.")
print("regional_warming.csv stays as-is (AR6 Atlas regional values).")

## Approach 2

In [27]:
import requests
import pandas as pd
from io import StringIO

url = "https://raw.githubusercontent.com/mathause/cmip_temperatures/main/cmip6_mmm_temperatures_one_ens_1850_1900.csv"
r = requests.get(url, timeout=15)
print(r.status_code)
print(r.text[:500])

200
period,SSP1-1.9 (14),SSP1-2.6 (41),SSP2-4.5 (42),SSP3-7.0 (35),SSP5-8.5 (44)
2021-2040,1.53,1.61,1.64,1.6,1.73
2041-2060,1.67,1.95,2.19,2.31,2.63
2081-2100,1.53,2.06,3.01,4.04,4.99
2016-2035,1.42,1.48,1.49,1.44,1.54
2046-2065,1.67,1.99,2.33,2.51,2.88
2011-2020,1.13,1.18,1.19,1.15,1.19
2021-2030,1.43,1.5,1.5,1.44,1.54
2031-2040,1.63,1.72,1.78,1.77,1.92
2041-2050,1.69,1.9,2.05,2.11,2.38
2051-2060,1.66,1.99,2.34,2.51,2.88
2061-2070,1.64,2.05,2.54,2.92,3.43
2071-2080,1.6,2.08,2.76,3.35,4.02
2081-209


In [28]:
import pandas as pd
import numpy as np
import requests
from io import StringIO

# ── Real CMIP6 multi-model mean temperatures
# Source: Mathias Hauser, ETH Zurich — pre-computed from raw CMIP6 output
# vs 1850-1900 baseline, CC BY-SA 4.0
# https://github.com/mathause/cmip_temperatures

url = "https://raw.githubusercontent.com/mathause/cmip_temperatures/main/cmip6_mmm_temperatures_one_ens_1850_1900.csv"
r = requests.get(url, timeout=15)
df = pd.read_csv(StringIO(r.text))
print(df)

       period  SSP1-1.9 (14)  SSP1-2.6 (41)  SSP2-4.5 (42)  SSP3-7.0 (35)  \
0   2021-2040           1.53           1.61           1.64           1.60   
1   2041-2060           1.67           1.95           2.19           2.31   
2   2081-2100           1.53           2.06           3.01           4.04   
3   2016-2035           1.42           1.48           1.49           1.44   
4   2046-2065           1.67           1.99           2.33           2.51   
5   2011-2020           1.13           1.18           1.19           1.15   
6   2021-2030           1.43           1.50           1.50           1.44   
7   2031-2040           1.63           1.72           1.78           1.77   
8   2041-2050           1.69           1.90           2.05           2.11   
9   2051-2060           1.66           1.99           2.34           2.51   
10  2061-2070           1.64           2.05           2.54           2.92   
11  2071-2080           1.60           2.08           2.76           3.35   

In [29]:
# ── Map column names to our scenario keys
# The file has 10-year period means as anchor points
scenario_col = {
    'ssp126': 'SSP1-2.6 (41)',
    'ssp245': 'SSP2-4.5 (42)',
    'ssp585': 'SSP5-8.5 (44)',
}

# ── Anchor periods we'll use (decadal midpoints)
# period → midyear
period_midyear = {
    '2021-2040': 2030,
    '2041-2060': 2050,
    '2061-2070': 2065,
    '2071-2080': 2075,
    '2081-2090': 2085,
    '2091-2100': 2095,
}

# ── Also need a 2015 anchor — use 2011-2020 period midpoint (2015~2016)
# and the historical observed value at ~2014 as starting point
# Pull NASA GISTEMP as before for historical
GISTEMP_URL = "https://data.giss.nasa.gov/gistemp/tabledata_v4/GLB.Ts+dSST.csv"
hist_raw = pd.read_csv(GISTEMP_URL, skiprows=1, na_values='****')
hist = hist_raw[['Year', 'J-D']].copy()
hist.columns = ['year', 'anomaly']
hist['anomaly'] = pd.to_numeric(hist['anomaly'], errors='coerce')
hist = hist.dropna()
hist['year'] = hist['year'].astype(int)
hist['anomaly'] = (hist['anomaly'] + 0.69).round(3)
hist.to_csv('data/historical.csv', index=False)
print(f"\nhistorical.csv: {len(hist)} rows")


historical.csv: 146 rows


In [30]:
last_year = int(hist['year'].max())
last_val = float(hist.loc[hist['year'] == last_year, 'anomaly'].values[0])
print(f"Last observed: {last_year} = {last_val}°C")

# Get 2015 value from historical data for smooth handoff
val_2015 = float(hist.loc[hist['year'] == 2015, 'anomaly'].values[0])
print(f"2015 observed: {val_2015}°C")

# ── Build projections.csv from CMIP6 anchor points
rows = []
for scenario, col in scenario_col.items():
    # Build anchor list: start from last observed year
    anchors = {2015: val_2015}
    
    for period, midyear in period_midyear.items():
        val = float(df.loc[df['period'] == period, col].values[0])
        anchors[midyear] = val
    
    anchor_years = sorted(anchors.keys())
    anchor_vals = [anchors[y] for y in anchor_years]
    
    # Interpolate annually
    all_years = list(range(2015, 2101))
    interp_vals = np.interp(all_years, anchor_years, anchor_vals)
    
    # Uncertainty band: use spread across models
    # SSP1-2.6: ±0.3→0.5, SSP2-4.5: ±0.3→0.7, SSP5-8.5: ±0.3→1.0
    # Scale linearly from 2025 to 2095
    unc_start = 0.3
    unc_end = {'ssp126': 0.5, 'ssp245': 0.7, 'ssp585': 1.0}[scenario]
    
    for year, median in zip(all_years, interp_vals):
        t = max(0, min(1, (year - last_year) / (2095 - last_year)))
        unc = unc_start + (unc_end - unc_start) * t
        rows.append({
            'year': year,
            'scenario': scenario,
            'median': round(median, 3),
            'likely_low': round(median - unc, 3),
            'likely_high': round(median + unc, 3),
        })

proj = pd.DataFrame(rows)
proj.to_csv('data/projections.csv', index=False)
print(f"\nprojections.csv: {len(proj)} rows")
print(proj[proj['year'].isin([2030, 2050, 2090])].to_string())

Last observed: 2025 = 1.88°C
2015 observed: 1.59°C

projections.csv: 258 rows
     year scenario  median  likely_low  likely_high
15   2030   ssp126   1.610       1.296        1.924
35   2050   ssp126   1.950       1.579        2.321
75   2090   ssp126   2.055       1.569        2.541
101  2030   ssp245   1.640       1.311        1.969
121  2050   ssp245   2.190       1.747        2.633
161  2090   ssp245   3.010       2.339        3.681
187  2030   ssp585   1.730       1.380        2.080
207  2050   ssp585   2.630       2.080        3.180
247  2090   ssp585   4.990       4.040        5.940


In [31]:
hist.to_csv('data/historical.csv', index=False)
proj.to_csv('data/projections.csv', index=False)
print("done")

done


## Temperature Data

In [34]:
import intake
import pandas as pd

col = intake.open_esm_datastore("https://storage.googleapis.com/cmip6/pangeo-cmip6.json")

# Search for pr — same models we know work
query = col.search(
    variable_id='pr',
    table_id='Amon',
    member_id='r1i1p1f1',
    grid_label='gn',
    experiment_id=['historical', 'ssp126', 'ssp245', 'ssp585'],
    source_id=['ACCESS-CM2', 'MPI-ESM1-2-LR', 'CanESM5']
)

print(f"Results: {len(query.df)}")
print(query.df[['source_id', 'experiment_id', 'zstore']])

Results: 12
        source_id experiment_id  \
0         CanESM5        ssp585   
1         CanESM5    historical   
2         CanESM5        ssp245   
3         CanESM5        ssp126   
4   MPI-ESM1-2-LR        ssp245   
5   MPI-ESM1-2-LR    historical   
6   MPI-ESM1-2-LR        ssp585   
7   MPI-ESM1-2-LR        ssp126   
8      ACCESS-CM2        ssp245   
9      ACCESS-CM2    historical   
10     ACCESS-CM2        ssp585   
11     ACCESS-CM2        ssp126   

                                               zstore  
0   gs://cmip6/CMIP6/ScenarioMIP/CCCma/CanESM5/ssp...  
1   gs://cmip6/CMIP6/CMIP/CCCma/CanESM5/historical...  
2   gs://cmip6/CMIP6/ScenarioMIP/CCCma/CanESM5/ssp...  
3   gs://cmip6/CMIP6/ScenarioMIP/CCCma/CanESM5/ssp...  
4   gs://cmip6/CMIP6/ScenarioMIP/MPI-M/MPI-ESM1-2-...  
5   gs://cmip6/CMIP6/CMIP/MPI-M/MPI-ESM1-2-LR/hist...  
6   gs://cmip6/CMIP6/ScenarioMIP/MPI-M/MPI-ESM1-2-...  
7   gs://cmip6/CMIP6/ScenarioMIP/MPI-M/MPI-ESM1-2-...  
8   gs://cmip6/CMIP6/Scenari

In [36]:
results_pr = {}  # {model: {experiment: DataArray}}

for _, row in query.df.iterrows():
    model = row['source_id']
    exp = row['experiment_id']
    zstore = row['zstore']
    
    print(f"Pulling {model} {exp}...")
    try:
        da = get_global_annual_mean(zstore, variable='pr')
        # Convert kg m-2 s-1 → mm/day
        da = da * 86400
        if model not in results_pr:
            results_pr[model] = {}
        results_pr[model][exp] = da
        print(f"  OK: {int(da.time.dt.year.min())}–{int(da.time.dt.year.max())}")
    except Exception as e:
        print(f"  ERROR: {e}")

print(f"\nDone. {len(results_pr)} models loaded.")

Pulling CanESM5 ssp585...


/var/folders/5y/1wpgx5fx0_53thqfv1v5x8tm0000gn/T/ipykernel_99603/3431655844.py:11: SerializationWarning: Unable to decode time axis into full numpy.datetime64[ns] objects, continuing using cftime.datetime objects instead, reason: dates out of range. To silence this warning use a coarser resolution 'time_unit' or specify 'use_cftime=True'.
  ds = xr.open_zarr(store, consolidated=True)


  OK: 2015–2300
Pulling CanESM5 historical...
  OK: 1850–2014
Pulling CanESM5 ssp245...
  OK: 2015–2100
Pulling CanESM5 ssp126...


/var/folders/5y/1wpgx5fx0_53thqfv1v5x8tm0000gn/T/ipykernel_99603/3431655844.py:11: SerializationWarning: Unable to decode time axis into full numpy.datetime64[ns] objects, continuing using cftime.datetime objects instead, reason: dates out of range. To silence this warning use a coarser resolution 'time_unit' or specify 'use_cftime=True'.
  ds = xr.open_zarr(store, consolidated=True)


  OK: 2015–2300
Pulling MPI-ESM1-2-LR ssp245...
  OK: 2015–2100
Pulling MPI-ESM1-2-LR historical...
  OK: 1850–2014
Pulling MPI-ESM1-2-LR ssp585...
  OK: 2015–2100
Pulling MPI-ESM1-2-LR ssp126...
  OK: 2015–2100
Pulling ACCESS-CM2 ssp245...
  OK: 2015–2100
Pulling ACCESS-CM2 historical...
  OK: 1850–2014
Pulling ACCESS-CM2 ssp585...


/var/folders/5y/1wpgx5fx0_53thqfv1v5x8tm0000gn/T/ipykernel_99603/3431655844.py:11: SerializationWarning: Unable to decode time axis into full numpy.datetime64[ns] objects, continuing using cftime.datetime objects instead, reason: dates out of range. To silence this warning use a coarser resolution 'time_unit' or specify 'use_cftime=True'.
  ds = xr.open_zarr(store, consolidated=True)
/var/folders/5y/1wpgx5fx0_53thqfv1v5x8tm0000gn/T/ipykernel_99603/3431655844.py:11: SerializationWarning: Unable to decode time axis into full numpy.datetime64[ns] objects, continuing using cftime.datetime objects instead, reason: dates out of range. To silence this warning use a coarser resolution 'time_unit' or specify 'use_cftime=True'.
  ds = xr.open_zarr(store, consolidated=True)
/var/folders/5y/1wpgx5fx0_53thqfv1v5x8tm0000gn/T/ipykernel_99603/3431655844.py:11: SerializationWarning: Unable to decode time axis into full numpy.datetime64[ns] objects, continuing using cftime.datetime objects instead, reas

  OK: 2015–2300
Pulling ACCESS-CM2 ssp126...


/var/folders/5y/1wpgx5fx0_53thqfv1v5x8tm0000gn/T/ipykernel_99603/3431655844.py:11: SerializationWarning: Unable to decode time axis into full numpy.datetime64[ns] objects, continuing using cftime.datetime objects instead, reason: dates out of range. To silence this warning use a coarser resolution 'time_unit' or specify 'use_cftime=True'.
  ds = xr.open_zarr(store, consolidated=True)
/var/folders/5y/1wpgx5fx0_53thqfv1v5x8tm0000gn/T/ipykernel_99603/3431655844.py:11: SerializationWarning: Unable to decode time axis into full numpy.datetime64[ns] objects, continuing using cftime.datetime objects instead, reason: dates out of range. To silence this warning use a coarser resolution 'time_unit' or specify 'use_cftime=True'.
  ds = xr.open_zarr(store, consolidated=True)
/var/folders/5y/1wpgx5fx0_53thqfv1v5x8tm0000gn/T/ipykernel_99603/3431655844.py:11: SerializationWarning: Unable to decode time axis into full numpy.datetime64[ns] objects, continuing using cftime.datetime objects instead, reas

  OK: 2015–2300

Done. 3 models loaded.


In [39]:
def get_years(time_values):
    try:
        return pd.DatetimeIndex(time_values).year.values
    except Exception:
        return np.array([int(t.year) for t in time_values])

rows_pr = []

for model, exps in results_pr.items():
    if 'historical' not in exps:
        continue
    
    hist_pr = exps['historical']
    hist_years = get_years(hist_pr.time.values)
    baseline_mask = (hist_years >= 1850) & (hist_years <= 1900)
    baseline_pr = float(hist_pr.values[baseline_mask].mean())
    print(f"{model} baseline pr: {baseline_pr:.4f} mm/day")
    
    for exp in ['ssp126', 'ssp245', 'ssp585']:
        if exp not in exps:
            continue
        proj_pr = exps[exp]
        proj_years = get_years(proj_pr.time.values)
        
        for year, val in zip(proj_years, proj_pr.values):
            if 2015 <= int(year) <= 2100:
                pct_change = round((float(val) - baseline_pr) / baseline_pr * 100, 3)
                rows_pr.append({
                    'model': model,
                    'scenario': exp,
                    'year': int(year),
                    'pr_mmday': round(float(val), 5),
                    'pr_pct_change': pct_change
                })

pr_df = pd.DataFrame(rows_pr)

pr_out = pr_df.groupby(['scenario', 'year']).agg(
    pr_mmday=('pr_mmday', 'median'),
    pr_pct_change=('pr_pct_change', 'median')
).reset_index().round(4)

pr_out.to_csv('data/precipitation.csv', index=False)
print(f"\nprecipitation.csv: {len(pr_out)} rows")
print(pr_out[pr_out['year'].isin([2030, 2050, 2090])].to_string())

CanESM5 baseline pr: 2.8593 mm/day
MPI-ESM1-2-LR baseline pr: 2.8293 mm/day
ACCESS-CM2 baseline pr: 3.1233 mm/day

precipitation.csv: 258 rows
    scenario  year  pr_mmday  pr_pct_change
15    ssp126  2030    2.9567          2.993
35    ssp126  2050    2.9881          3.995
75    ssp126  2090    3.0245          5.778
101   ssp245  2030    2.9262          2.249
121   ssp245  2050    2.9778          3.835
161   ssp245  2090    3.0730          6.659
187   ssp585  2030    2.9556          2.551
207   ssp585  2050    3.0284          4.708
247   ssp585  2090    3.2162          9.502
